# Zone-wise Weather vs Demand Analysis

This notebook follows the same analysis pattern as your script and is organized so each step can be run one-by-one.

## 1) Imports & Configuration
Set `selected_zone`, `selected_regions`, file paths, and cleaning options.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# -----------------------------
# User Parameters
# -----------------------------
selected_zone = "TPSODL"  # Options: TPCODL, TPNODL, TPSODL, TPWODL
selected_regions = []      # Example: ["TPCODL_ANGUL", "TPCODL_CUTTACK"]; keep [] for all regions in selected_zone
# set to True to run the notebook using the overall total demand column
# and all weather observations (ignore zone/region filters)
use_total_demand = False

weather_dir = Path("weather_by_zone")
demand_file = Path("cleaned_demand_data.csv")

# Columns to remove (edit as needed)
columns_to_drop = [
    "temperature_80m (undefined)",
    "soil_temperature_0cm (undefined)",
]

# Missing value handling
feature_fill_method = "interpolate_then_median"  # options: interpolate_then_median, median_only

WEATHER_FEATURES = ["temperature", "humidity", "precipitation", "wind_speed"]
DEMAND_ZONE_COLUMN_MAP = {
    "TPCODL": "TPCODL Demand",
    "TPNODL": "TPNODL Demand",
    "TPSODL": "TPSOSDL Demand",
    "TPWODL": "TPWODL Demand",
}
TOTAL_DEMAND_COL = "Total Demand (as recorded)"

if not use_total_demand:
    assert selected_zone in DEMAND_ZONE_COLUMN_MAP, f"Invalid zone: {selected_zone}"
else:
    # override label for naming and disable region filtering
    selected_zone = "TOTAL"
    selected_regions = []

print("Configuration loaded")
print(f"selected_zone: {selected_zone}")
print(f"use_total_demand: {use_total_demand}")
print(f"weather_dir: {weather_dir.resolve()}")
print(f"demand_file: {demand_file.resolve()}")

## 2) Utility Functions

In [ ]:
def _norm_col(c: str) -> str:
    return " ".join(c.strip().split())


def _canonical_col_key(c: str) -> str:
    c = c.lower().strip()
    c = c.replace("(a", "(")
    return c


def map_weather_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Robust mapping for clean/mojibake header variants
    rename_map = {}
    for col in df.columns:
        k = _canonical_col_key(col)
        if "temperature_2m" in k:
            rename_map[col] = "temperature"
        elif "relative_humidity_2m" in k:
            rename_map[col] = "humidity"
        elif k.startswith("rain") or "rain (mm)" in k:
            rename_map[col] = "precipitation"
        elif "wind_speed_10m" in k:
            rename_map[col] = "wind_speed"
    return df.rename(columns=rename_map)


def safe_corr(a: pd.Series, b: pd.Series, min_points: int = 3) -> float:
    pair = pd.concat([a, b], axis=1).dropna()
    if len(pair) < min_points:
        return np.nan
    a0 = pair.iloc[:, 0]
    b0 = pair.iloc[:, 1]
    if a0.nunique(dropna=True) <= 1 or b0.nunique(dropna=True) <= 1:
        return np.nan
    sa = a0.std()
    sb = b0.std()
    if pd.isna(sa) or pd.isna(sb) or sa == 0 or sb == 0:
        return np.nan
    return a0.corr(b0)


def calculate_lead_lag_correlation(weather_series: pd.Series, demand_series: pd.Series, max_lag: int = 12) -> pd.Series:
    correlations = {}
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            corr = safe_corr(weather_series.iloc[:lag], demand_series.iloc[-lag:])
        elif lag > 0:
            corr = safe_corr(weather_series.iloc[lag:], demand_series.iloc[:-lag])
        else:
            corr = safe_corr(weather_series, demand_series)
        correlations[lag] = corr
    return pd.Series(correlations)


def detect_changepoints(series: pd.Series, window: int = 24, threshold_std: float = 1.5):
    changes = series.diff().fillna(0)
    rolling_std = changes.rolling(window=window, min_periods=1).std()
    rolling_mean = changes.rolling(window=window, min_periods=1).mean()
    changepoints = (changes - rolling_mean).abs() > (threshold_std * rolling_std)
    return changepoints.fillna(False), changes, rolling_std


def sliding_window_correlation(weather_change: pd.Series, demand_change: pd.Series, window_size: int = 168) -> pd.Series:
    if len(weather_change) <= window_size:
        return pd.Series(dtype=float)
    corrs = []
    idx = []
    for i in range(len(weather_change) - window_size):
        c = safe_corr(weather_change.iloc[i:i+window_size], demand_change.iloc[i:i+window_size])
        corrs.append(c)
        idx.append(i + window_size // 2)
    return pd.Series(corrs, index=idx)



## 3) Data Loading
Loads all weather files and demand data.

In [ ]:
weather_files = sorted(weather_dir.glob("weather_*.csv"))
print(f"Found weather files: {len(weather_files)}")

weather_parts = []
for fp in weather_files:
    tmp = pd.read_csv(fp)
    tmp.columns = [_norm_col(c) for c in tmp.columns]
    tmp = map_weather_columns(tmp)

    if "zone" not in tmp.columns or "time" not in tmp.columns:
        print(f"Skipping malformed file: {fp.name}")
        continue

    tmp["source_file"] = fp.name
    tmp["time"] = pd.to_datetime(tmp["time"], errors="coerce")
    tmp["discom_zone"] = tmp["zone"].astype(str).str.split("_").str[0]
    weather_parts.append(tmp)

weather_raw = pd.concat(weather_parts, ignore_index=True)

# Demand
demand_raw = pd.read_csv(demand_file)
demand_raw["Timestamp"] = pd.to_datetime(demand_raw["Timestamp"], errors="coerce")

print("weather_raw shape:", weather_raw.shape)
print("demand_raw shape:", demand_raw.shape)

## 4) Initial Data Inspection
Check shape, columns, dtypes, summary, and missing-value patterns.

In [ ]:
print("Weather shape:", weather_raw.shape)
print("Demand shape:", demand_raw.shape)

print("Weather columns:")
print(weather_raw.columns.tolist())

print("Demand columns:")
print(demand_raw.columns.tolist())

print("Weather dtypes:")
print(weather_raw.dtypes)

print("Demand dtypes:")
print(demand_raw.dtypes)

print("Weather describe:")
print(weather_raw.describe(include="all").T.head(25))

print("Demand describe:")
print(demand_raw.describe(include="all").T)

weather_nan = weather_raw.isna().sum().sort_values(ascending=False)
demand_nan = demand_raw.isna().sum().sort_values(ascending=False)

print("Top missing columns in weather:")
print(weather_nan.head(20))

print("Missing columns in demand:")
print(demand_nan[demand_nan > 0])

## 5) Column Removal (Editable)
Use this cell to remove unnecessary columns before cleaning.

In [ ]:
weather = weather_raw.copy()
existing_drop_cols = [c for c in columns_to_drop if c in weather.columns]

print("Columns requested to drop:", columns_to_drop)
print("Columns actually dropped:", existing_drop_cols)

weather = weather.drop(columns=existing_drop_cols, errors="ignore")
print("Shape after column removal:", weather.shape)

## 6) NaN Cleaning
1. Remove fully empty rows (two such rows expected in your note).
2. Remove rows with missing `time`/`zone`.
3. Clean weather feature NaNs by group (`location_id`) using interpolation + median fill.

In [ ]:
# 1) Drop fully empty rows
before_rows = len(weather)
weather = weather.dropna(how="all")
removed_full_nan = before_rows - len(weather)

# 2) Remove rows missing critical identifiers
weather = weather.dropna(subset=["time", "zone"]).copy()

# Ensure required feature columns exist
for f in WEATHER_FEATURES:
    if f not in weather.columns:
        weather[f] = np.nan
    weather[f] = pd.to_numeric(weather[f], errors="coerce")

# Additional safeguard: if all key weather features are NaN, drop row
all_feat_nan = weather[WEATHER_FEATURES].isna().all(axis=1)
removed_all_feature_nan = int(all_feat_nan.sum())
weather = weather.loc[~all_feat_nan].copy()

# 3) Fill remaining NaNs per location
if "location_id" not in weather.columns:
    weather["location_id"] = -1

weather = weather.sort_values(["location_id", "time"]).copy()

if feature_fill_method == "interpolate_then_median":
    for f in WEATHER_FEATURES:
        weather[f] = weather.groupby("location_id")[f].transform(lambda s: s.interpolate(limit_direction="both"))
        weather[f] = weather.groupby("location_id")[f].transform(lambda s: s.fillna(s.median()))
else:
    for f in WEATHER_FEATURES:
        weather[f] = weather.groupby("location_id")[f].transform(lambda s: s.fillna(s.median()))

# Final fallback: global median
for f in WEATHER_FEATURES:
    weather[f] = weather[f].fillna(weather[f].median())

# Time alignment key
weather["timestamp_rounded"] = pd.to_datetime(weather["time"]).dt.floor("h")
demand = demand_raw.dropna(subset=["Timestamp"]).copy()
demand["timestamp_rounded"] = demand["Timestamp"].dt.floor("h")

print(f"Removed fully empty weather rows: {removed_full_nan}")
print(f"Removed rows with all 4 weather features NaN: {removed_all_feature_nan}")
print("Remaining weather NaNs (core features):")
print(weather[WEATHER_FEATURES].isna().sum())
print("Cleaned weather shape:", weather.shape)

## 7) Zone Filtering (and Optional Region Filtering)
This step keeps one zone and optionally specific regions.

In [ ]:
if use_total_demand:
    # keep all weather records when analysing total demand
    zone_weather = weather.copy()
else:
    zone_weather = weather[weather["discom_zone"] == selected_zone].copy()
    if selected_regions:
        zone_weather = zone_weather[zone_weather["zone"].isin(selected_regions)].copy()

label = "all zones" if use_total_demand else f"zone {selected_zone}"
print(f"Weather shape ({label}):", zone_weather.shape)
print(f"Regions in selection:", zone_weather["zone"].nunique())
print("Sample regions:", sorted(zone_weather["zone"].dropna().unique().tolist())[:10])

## 8) Prepare Demand Series for Selected Zone

In [ ]:
if use_total_demand:
    demand_col = TOTAL_DEMAND_COL
else:
    demand_col = DEMAND_ZONE_COLUMN_MAP[selected_zone]
    if demand_col not in demand.columns:
        raise ValueError(f"Demand column not found for zone {selected_zone}: {demand_col}")

# convert to numeric and aggregate
if demand_col not in demand.columns:
    raise ValueError(f"Demand column not found: {demand_col}")

demand["zone_demand"] = pd.to_numeric(demand[demand_col], errors="coerce")
demand_zone_hourly = demand.groupby("timestamp_rounded", as_index=True)["zone_demand"].mean().sort_index()

print("Demand points:", len(demand_zone_hourly))
print("Demand date range:", demand_zone_hourly.index.min(), "to", demand_zone_hourly.index.max())

In [ ]:
# Cell A: Analysis setup (insert after merge step)

from pathlib import Path
import re


# Fallback from your existing pipeline variables
demand_tmp = demand_zone_hourly.rename("demand").reset_index()
wx_cols = ["timestamp_rounded", "zone", "location_id", "temperature", "humidity", "precipitation", "wind_speed"]
analysis_df = zone_weather[wx_cols].merge(demand_tmp, on="timestamp_rounded", how="inner")

# Normalize timestamp and numeric fields
if "timestamp_rounded" not in analysis_df.columns:
    if "time" in analysis_df.columns:
        analysis_df["timestamp_rounded"] = pd.to_datetime(analysis_df["time"], errors="coerce").dt.floor("h")
    else:
        raise ValueError("No timestamp column found. Expected `timestamp_rounded` or `time`.")

analysis_df["timestamp_rounded"] = pd.to_datetime(analysis_df["timestamp_rounded"], errors="coerce")
analysis_df["demand"] = pd.to_numeric(analysis_df["demand"], errors="coerce")
for c in ["temperature", "humidity", "precipitation", "wind_speed"]:
    analysis_df[c] = pd.to_numeric(analysis_df[c], errors="coerce")

analysis_df = analysis_df.dropna(subset=["timestamp_rounded", "zone", "location_id", "demand"]).copy()
analysis_df = analysis_df.sort_values("timestamp_rounded")

# Demand is zone-level and repeated across weather rows: build once from unique hourly demand
zone_demand_ts = (
    analysis_df.groupby("timestamp_rounded", as_index=True)["demand"]
    .mean()
    .sort_index()
)

analysis_dir = Path("Analysis")
analysis_dir.mkdir(parents=True, exist_ok=True)

def _safe_name(x: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(x)).strip("_")

def _savefig(name: str, dpi: int = 150):
    plt.tight_layout()
    plt.savefig(analysis_dir / name, dpi=dpi, bbox_inches="tight")
    plt.close()

print(f"Analysis rows: {len(analysis_df):,}")
print(f"Unique regions: {analysis_df['zone'].nunique()}")
print(f"Zone demand points: {len(zone_demand_ts):,}")
print(f"Analysis directory: {analysis_dir.resolve()}")


## 9) Interactive Global Demand Plots
The following cell replaces the static plots with interactive Plotly versions. It uses the helper functions defined inline and writes HTML files to the `Analysis/` directory.

In [ ]:
# Plotly setup + helpers
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

analysis_dir = Path("Analysis")
analysis_dir.mkdir(parents=True, exist_ok=True)

def save_html(fig, filename, height=500, width=1100):
    fig.update_layout(
        template="plotly_white",
        height=height,
        width=width,
        margin=dict(l=50, r=30, t=70, b=50),
        hovermode="x unified",
    )
    out = analysis_dir / filename
    fig.write_html(str(out), include_plotlyjs="cdn")
    return out

# prepare dataframe for plotting
zd = zone_demand_ts.dropna().sort_index()
tmp = zd.rename("demand").reset_index().rename(columns={"timestamp_rounded": "time"})
tmp["hour"] = tmp["time"].dt.hour
tmp["dow"] = tmp["time"].dt.day_name()
tmp["month"] = tmp["time"].dt.to_period("M").dt.to_timestamp()

# 1) Demand time series
fig = px.line(tmp, x="time", y="demand", title=f"{selected_zone} Zone Demand Time Series")
save_html(fig, f"{selected_zone}_GLOBAL_demand_timeseries.html")

# 2) Demand distribution
fig = px.histogram(tmp, x="demand", nbins=50, marginal="box", title=f"{selected_zone} Demand Distribution")
save_html(fig, f"{selected_zone}_GLOBAL_demand_distribution.html")

# 3) Hour-of-day profile
hour_prof = tmp.groupby("hour", as_index=False)["demand"].mean()
fig = px.line(hour_prof, x="hour", y="demand", markers=True, title=f"{selected_zone} Average Demand by Hour")
save_html(fig, f"{selected_zone}_GLOBAL_hourly_profile.html")

# 4) DOW x Hour heatmap
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
heat = tmp.pivot_table(index="dow", columns="hour", values="demand", aggfunc="mean").reindex(day_order)
fig = px.imshow(
    heat,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    labels=dict(x="Hour", y="Day of Week", color="Mean Demand"),
    title=f"{selected_zone} Demand Heatmap (Day vs Hour)"
)
save_html(fig, f"{selected_zone}_GLOBAL_dow_hour_heatmap.html")

# 5) Monthly trend
monthly = tmp.groupby("month", as_index=False)["demand"].mean()
fig = px.line(monthly, x="month", y="demand", markers=True, title=f"{selected_zone} Monthly Demand Trend")
save_html(fig, f"{selected_zone}_GLOBAL_monthly_trend.html")

print("Saved interactive global plots to Analysis/")

## 10) Interactive Region Plots
Loop over each zone/region to produce Plotly HTML outputs and temperature bucket tables.

In [ ]:
import time
from sklearn.preprocessing import StandardScaler

weather_features = ["temperature", "humidity", "precipitation", "wind_speed"]

regions_out = []
summary_stats_long = []
corr_long = []

region_hourly = (
    analysis_df
    .groupby(["zone", "location_id", "timestamp_rounded"], as_index=False)[weather_features]
    .mean()
)

region_hourly = region_hourly.sort_values(["zone","location_id","timestamp_rounded"])

for (region_name, location_id), g in region_hourly.groupby(["zone", "location_id"]):
    region_start = time.perf_counter()
    print(f"\n--- Processing region: {region_name} | location_id: {location_id} ---")

    g = g.set_index("timestamp_rounded")
    aligned = g.join(zone_demand_ts.rename("demand"), how="inner")
    print(f"Alignment time: {time.perf_counter() - region_start:.3f} sec")

    if len(aligned) < 50:
        print("Skipped (insufficient data)")
        continue

    region_key = _safe_name(region_name)
    base = f"{selected_zone}_{region_key}"

    # ----------------------------
    # BASIC STATISTICS
    # ----------------------------
    t0 = time.perf_counter()
    for v in ["demand"] + weather_features:
        s = aligned[v].dropna()
        summary_stats_long.append({
            "selected_zone": selected_zone,
            "region": region_name,
            "location_id": location_id,
            "variable": v,
            "mean": s.mean(),
            "std": s.std(),
            "skew": s.skew(),
            "kurtosis": s.kurt()
        })
    print(f"Summary stats time: {time.perf_counter() - t0:.3f} sec")

    # ----------------------------
    # CORRELATION
    # ----------------------------
    t0 = time.perf_counter()
    for feat in weather_features:
        pair = aligned[["demand", feat]].dropna()
        if len(pair) < 3:
            pearson_val = np.nan
            spearman_val = np.nan
        else:
            pearson_val = pair["demand"].corr(pair[feat], method="pearson")
            spearman_val = pair["demand"].corr(pair[feat], method="spearman")
        corr_long.append({
            "selected_zone": selected_zone,
            "region": region_name,
            "location_id": location_id,
            "feature": feat,
            "pearson_corr": pearson_val,
            "spearman_corr": spearman_val
        })
    print(f"Correlation time: {time.perf_counter() - t0:.3f} sec")

    # ----------------------------
    # NORMALIZE + DIFFERENCE
    # ----------------------------
    t0 = time.perf_counter()
    scaler = StandardScaler()
    norm_df = aligned[["demand"] + weather_features].dropna()
    norm_values = scaler.fit_transform(norm_df)
    norm_df = pd.DataFrame(norm_values, index=norm_df.index, columns=["demand"] + weather_features)
    diff_df = norm_df.diff().dropna()
    diff_df.columns = [
        "demand_diff",
        "temp_diff",
        "humidity_diff",
        "precip_diff",
        "wind_diff"
    ]
    print(f"Normalization + diff time: {time.perf_counter() - t0:.3f} sec")

    # ----------------------------
    # INTERACTIVE PLOTLY OUTPUTS
    # ----------------------------
    # diff correlation heatmap
    corr = diff_df.corr()
    fig = px.imshow(
        corr, text_auto=".2f", aspect="auto", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
        title=f"{selected_zone} | {region_name} | Diff Correlation Heatmap"
    )
    save_html(fig, f"{base}_diff_correlation.html")

    # diff series
    fig = go.Figure()
    for c in diff_df.columns:
        fig.add_trace(go.Scatter(x=diff_df.index, y=diff_df[c], mode="lines", name=c))
    fig.update_layout(title=f"{selected_zone} | {region_name} | First-Order Difference Series", yaxis_title="Difference")
    save_html(fig, f"{base}_diff_series.html", height=550)

    # temperature bucket curve
    d = aligned.copy()
    temp_bins = pd.cut(d["temperature"], bins=20, duplicates="drop")
    bucket = (
        d.assign(temp_bin=temp_bins)
         .dropna(subset=["temp_bin"])
         .groupby("temp_bin", observed=True)
         .agg(avg_temperature=("temperature","mean"), mean_demand=("demand","mean"), n=("demand","size"))
         .reset_index(drop=True)
    )
    bucket.to_csv(analysis_dir / f"{base}_temp_bucket_table.csv", index=False)
    fig = px.line(
        bucket, x="avg_temperature", y="mean_demand", markers=True,
        hover_data=["n"],
        title=f"{selected_zone} | {region_name} | Temperature Bucket Curve"
    )
    save_html(fig, f"{base}_temp_bucket_curve.html")

    # ----------------------------
    # REGION SUMMARY ROW
    # ----------------------------
    regions_out.append({
        "selected_zone": selected_zone,
        "region": region_name,
        "location_id": location_id,
        "n_points": len(aligned),
    })

    print(f"Total region runtime: {time.perf_counter() - region_start:.3f} sec")

# ----------------------------
# FINAL OUTPUT CSVS
# ----------------------------
region_overview_df = pd.DataFrame(regions_out)
region_corr_df = pd.DataFrame(corr_long)
region_stats_df = pd.DataFrame(summary_stats_long)

region_overview_df.to_csv(
    analysis_dir / f"{selected_zone}_region_analysis_overview.csv",
    index=False
)
region_corr_df.to_csv(
    analysis_dir / f"{selected_zone}_all_region_correlations.csv",
    index=False
)
region_stats_df.to_csv(
    analysis_dir / f"{selected_zone}_all_region_summary_stats.csv",
    index=False
)

print("\nSaved region-level analysis outputs.")
print(f"Regions analyzed: {len(region_overview_df)}")

In [ ]:
# Cell D: README-style summary report (console + file)

report_lines = []
report_lines.append(f"# Statistical Analysis Summary: {selected_zone}")
report_lines.append("")
report_lines.append("## 1) Analyses Performed")
report_lines.append("- Global demand analysis (time series, distribution, hour-of-day profile, day-of-week heatmap, monthly trend).")
report_lines.append("- Region-level descriptive statistics (mean, std, skew, kurtosis) for demand and weather features.")
report_lines.append("- Demand-weather correlation analysis (Pearson and Spearman) per region.")
report_lines.append("- First-order difference analysis with diff-correlation heatmaps and diff-series plots.")
report_lines.append("- Temperature bucket curve analysis (mean demand by temperature bins).")
report_lines.append("- Sliding window local correlation between temperature change and demand change.")
report_lines.append("- Demand change-point detection and visualization.")
report_lines.append("")

report_lines.append("## 2) Key Insights")
if len(region_corr_df) > 0:
    corr_valid = region_corr_df.dropna(subset=["pearson_corr"]).copy()
    if len(corr_valid) > 0:
        corr_valid["abs_pearson"] = corr_valid["pearson_corr"].abs()
        best = corr_valid.sort_values("abs_pearson", ascending=False).iloc[0]
        report_lines.append(
            f"- Strongest linear relationship: region `{best['region']}`, feature `{best['feature']}`, "
            f"Pearson={best['pearson_corr']:.3f}, Spearman="
            f"{region_corr_df.loc[best.name, 'spearman_corr']:.3f}."
        )

        avg_by_feat = corr_valid.groupby("feature")["pearson_corr"].mean().sort_values(key=lambda s: s.abs(), ascending=False)
        for feat, val in avg_by_feat.items():
            report_lines.append(f"- Average Pearson(demand, {feat}) across regions: {val:.3f}.")

        # additional region-level insights
        if 'region_overview_df' in globals():
            n_regions = len(region_overview_df)
            avg_points = region_overview_df['n_points'].mean()
            max_points = region_overview_df['n_points'].max()
            min_points = region_overview_df['n_points'].min()
            report_lines.append(f"- Data was analyzed for {n_regions} regions with an average of {avg_points:.1f} hourly points per region.")
            report_lines.append(f"- Region with most data had {max_points} points; least had {min_points}.")
            # optionally note low-sample regions
            sparse = region_overview_df[region_overview_df['n_points'] < 100]
            if len(sparse) > 0:
                report_lines.append(f"- {len(sparse)} regions had fewer than 100 data points; their statistics may be unreliable.")
    else:
        report_lines.append("- Correlation values were mostly unavailable after alignment/NaN filtering.")
else:
    report_lines.append("- No region-level correlation results were produced (likely insufficient aligned data points).")

# if len(region_overview_df) > 0:
#     # cp_mean = region_overview_df["n_demand_changepoints"].mean()
#     # swc_mean = region_overview_df["swc_temp_demand_mean"].mean()
#     # report_lines.append(f"- Average detected demand change points per region: {cp_mean:.1f}.")
#     # report_lines.append(f"- Mean sliding-window temp-demand diff correlation across regions: {swc_mean:.3f}.")
# # report_lines.append("")

report_lines.append("## 3) Suggested Additional Statistical Methods")
report_lines.append("- **Granger causality tests**: useful when testing whether past weather values improve demand forecasting.")
report_lines.append("- **Cross-correlation functions (CCF)**: useful to detect lag structure beyond a single best lag.")
report_lines.append("- **Seasonal decomposition (STL)**: useful to separate trend/seasonality/residual effects in demand.")
report_lines.append("- **Autocorrelation & partial autocorrelation (ACF/PACF)**: useful for identifying temporal dependence for time-series modeling.")
report_lines.append("- **Mutual information analysis**: useful for nonlinear dependence not captured by Pearson correlation.")
report_lines.append("- **Quantile regression**: useful when weather affects peak demand differently than median demand.")
report_lines.append("- **Extreme weather impact analysis**: useful for threshold effects (heatwaves/heavy rain/wind events).")
report_lines.append("- **Demand elasticity vs temperature**: useful for interpretable sensitivity estimates by temperature regime.")
report_lines.append("")

report_text = "\n".join(report_lines)
print(report_text)

summary_path = analysis_dir / "statistical_analysis_summary.md"
summary_path.write_text(report_text, encoding="utf-8")
print(f"\nSaved summary report: {summary_path.resolve()}")
